## 1. Data Ingestion

In [52]:
from datetime import datetime,timezone
import os
import numpy as np
import pandas as pd

### a. Bronze layer

In [10]:
# 1. directories setup
DOMAINS = ['flights', 'bookings', 'passengers', 'payments']
for domain in DOMAINS:
  os.makedirs(f'storage/bronze/{domain}', exist_ok=True)
  os.makedirs(f'storage/quarantine/{domain}', exist_ok=True)

In [11]:
excel_file_path = 'UseCase - Airlines.xlsx'
xls = pd.ExcelFile(excel_file_path)

In [12]:
# 2. data quality checking/entity

def dq_flights(df):
  null_pk = df['flight_id'].isnull() | (
      df['flight_id'].astype(str).str.strip() == ''
  )
  null_times = df['departure_time'].isnull() | df['arrival_time'].isnull()
  valid = ~null_pk & ~null_times

  e1 = np.where(null_pk, 'ERR_NULL_FLIGHT_ID; ', '')
  e2 = np.where(null_times, 'ERR_NULL_TIMESTAMPS; ', '')
  w1 = np.where(
      df['airline'].isnull() | (df['airline'] == 'UNKNOWN'),
      'WARN_MISSING_AIRLINE; ',
      '',
  )

  df['_is_valid'] = valid
  df['_ingestion_errors'] = (
      pd.Series(e1, index=df.index)
      + pd.Series(e2, index=df.index)
      + pd.Series(w1, index=df.index)
  ).str.strip('; ')
  return df

In [13]:
def dq_bookings(df):
  null_pk = df['booking_id'].isnull() | (
      df['booking_id'].astype(str).str.strip() == ''
  )
  invalid_status = df['status'] == 'INVALID'
  valid = ~null_pk & ~invalid_status

  e1 = np.where(null_pk, 'ERR_NULL_BOOKING_ID; ', '')
  e2 = np.where(invalid_status, 'ERR_INVALID_STATUS; ', '')
  w1 = np.where(df['status'].isnull(), 'WARN_NULL_STATUS; ', '')

  df['_is_valid'] = valid
  df['_ingestion_errors'] = (
      pd.Series(e1, index=df.index)
      + pd.Series(e2, index=df.index)
      + pd.Series(w1, index=df.index)
  ).str.strip('; ')
  return df

In [14]:
def dq_passengers(df):
  null_pk = df['passenger_id'].isnull() | (
      df['passenger_id'].astype(str).str.strip() == ''
  )
  valid = ~null_pk

  e1 = np.where(null_pk, 'ERR_NULL_PASSENGER_ID; ', '')
  w1 = np.where(df['last_name'].isnull(), 'WARN_MISSING_LAST_NAME; ', '')

  df['_is_valid'] = valid
  df['_ingestion_errors'] = (
      pd.Series(e1, index=df.index) + pd.Series(w1, index=df.index)
  ).str.strip('; ')
  return df

In [15]:
def dq_payments(df):
  null_pk = df['payment_id'].isnull() | (
      df['payment_id'].astype(str).str.strip() == ''
  )
  numeric_amount = pd.to_numeric(df['amount'], errors='coerce')
  invalid_amount = numeric_amount.isnull()

  valid = ~null_pk & ~invalid_amount
  e1 = np.where(null_pk, 'ERR_NULL_PAYMENT_ID; ', '')
  e2 = np.where(invalid_amount, 'ERR_INVALID_AMOUNT; ', '')

  df['_is_valid'] = valid
  df['_ingestion_errors'] = (
      pd.Series(e1, index=df.index) + pd.Series(e2, index=df.index)
  ).str.strip('; ')
  return df

In [53]:
# 3. ingestion engine
def process_domain_ingestion(sheet_name, dq_check_fn):
  df = pd.read_excel(xls, sheet_name=sheet_name)

  # Attach Data Lineage Metadata
  df['_ingested_at'] = datetime.now(timezone.utc).isoformat()
  df['_source_file'] = os.path.basename(excel_file_path)

  # Run DQ Check
  df = dq_check_fn(df)

  # Route Clean vs Corrupted Records
  clean_df = df[df['_is_valid']].copy()
  quarantine_df = df[~df['_is_valid']].copy()

  # Write to Storage Layer
  clean_df.to_csv(
      f'storage/bronze/{sheet_name}/{sheet_name}_bronze.csv', index=False
  )
  if not quarantine_df.empty:
    quarantine_df.to_csv(
        f'storage/quarantine/{sheet_name}/{sheet_name}_quarantine.csv',
        index=False,
    )

  return len(df), len(clean_df), len(quarantine_df)


# 4. EXECUTE FOR ALL 4 SHEETS
results = {
    'flights': process_domain_ingestion('flights', dq_flights),
    'bookings': process_domain_ingestion('bookings', dq_bookings),
    'passengers': process_domain_ingestion('passengers', dq_passengers),
    'payments': process_domain_ingestion('payments', dq_payments),
}

# 5. SUMMARY OUTPUT
summary_df = pd.DataFrame(
    results,
    index=[
        'Total Records Read',
        'Clean Records (Bronze Layer)',
        'Corrupted Records (Quarantined/DLQ)',
    ],
).T
print('MULTI-ENTITY INGESTION COMPLETED')
print(summary_df)

MULTI-ENTITY INGESTION COMPLETED
            Total Records Read  Clean Records (Bronze Layer)  \
flights                   1020                          1020   
bookings                  1000                           970   
passengers                1039                          1039   
payments                  1000                           922   

            Corrupted Records (Quarantined/DLQ)  
flights                                       0  
bookings                                     30  
passengers                                    0  
payments                                     78  


### b. Silver layer

In [17]:
import os
import pandas as pd

In [18]:
# 1. SETUP SILVER STORAGE FOLDERS
for entity in ['flights', 'passengers', 'bookings', 'payments']:
  os.makedirs(f'storage/silver/{entity}', exist_ok=True)

In [21]:
# 2. LOAD BRONZE DATASETS
bronze_flights = pd.read_csv('storage/bronze/flights/flights_bronze.csv')
bronze_passengers = pd.read_csv(
    'storage/bronze/passengers/passengers_bronze.csv')
bronze_bookings = pd.read_csv('storage/bronze/bookings/bookings_bronze.csv')
bronze_payments = pd.read_csv('storage/bronze/payments/payments_bronze.csv')

In [22]:
# 3. FLIGHTS SILVER TRANSFORMATION 
bronze_flights['departure_time'] = pd.to_datetime(
    bronze_flights['departure_time']
)
bronze_flights['arrival_time'] = pd.to_datetime(
    bronze_flights['arrival_time']
)


def impute_airline(row):
  airline = str(row['airline']).strip()
  flight_id = str(row['flight_id']).strip()
  if pd.isna(row['airline']) or airline in ['UNKNOWN', 'nan', '']:
    if flight_id.startswith('AI'):
      return 'Air India'
    elif flight_id.startswith('6F') or flight_id.startswith('6E'):
      return 'IndiGo'
    elif flight_id.startswith('SJ'):
      return 'SpiceJet'
    elif flight_id.startswith('UK'):
      return 'Vistara'
    else:
      return 'Other'
  return airline


bronze_flights['airline_clean'] = bronze_flights.apply(impute_airline, axis=1)
overnight_mask = (
    bronze_flights['arrival_time'] < bronze_flights['departure_time']
)
bronze_flights['arrival_time_clean'] = bronze_flights['arrival_time']
bronze_flights.loc[overnight_mask, 'arrival_time_clean'] += pd.Timedelta(
    days=1
)
bronze_flights['duration_minutes'] = (
    bronze_flights['arrival_time_clean'] - bronze_flights['departure_time']
).dt.total_seconds() / 60.0

silver_flights = bronze_flights.drop_duplicates(
    subset=['flight_id', 'departure_time'], keep='first'
).copy()
silver_flights['airline'] = silver_flights['airline_clean']
silver_flights['arrival_time'] = silver_flights['arrival_time_clean']
silver_flights = silver_flights[[
    'flight_id',
    'airline',
    'source',
    'destination',
    'departure_time',
    'arrival_time',
    'duration_minutes',
    '_ingested_at',
    '_source_file',
]]

In [23]:
#  4. PASSENGERS SILVER TRANSFORMATION 
bronze_passengers['last_name'] = bronze_passengers['last_name'].fillna('Unknown')
bronze_passengers['date_of_birth'] = pd.to_datetime(
    bronze_passengers['date_of_birth']
)
bronze_passengers['age'] = (
    pd.to_datetime('2026-09-05') - bronze_passengers['date_of_birth']
).dt.days // 365
bronze_passengers['aadhaar_id'] = (
    bronze_passengers['aadhaar_id'].astype(str).str.zfill(12)
)
silver_passengers = bronze_passengers.drop_duplicates(
    subset=['passenger_id'], keep='first'
).copy()
silver_passengers = silver_passengers[[
    'passenger_id',
    'first_name',
    'last_name',
    'age',
    'gender',
    'email',
    'phone',
    'aadhaar_id',
    'date_of_birth',
    '_ingested_at',
    '_source_file',
]]

In [24]:
#  5. BOOKINGS SILVER TRANSFORMATION 
silver_bookings = bronze_bookings[
    bronze_bookings['status'] != 'INVALID'
].copy()
silver_bookings['status'] = silver_bookings['status'].fillna('PENDING')
silver_bookings['booking_date'] = pd.to_datetime(
    silver_bookings['booking_date']
)
silver_bookings = silver_bookings.drop_duplicates(
    subset=['booking_id'], keep='first'
)
silver_bookings = silver_bookings[[
    'booking_id',
    'passenger_id',
    'flight_id',
    'booking_date',
    'status',
    'passport_number',
    'seat_number',
    'emergency_contact_name',
    'emergency_contact_phone',
    '_ingested_at',
    '_source_file',
]]



In [25]:
#  6. PAYMENTS SILVER TRANSFORMATION 
bronze_payments['amount_numeric'] = pd.to_numeric(
    bronze_payments['amount'], errors='coerce'
)
silver_payments = bronze_payments[
    ~bronze_payments['amount_numeric'].isnull()
].copy()
silver_payments['amount'] = silver_payments['amount_numeric']
silver_payments = silver_payments.drop_duplicates(
    subset=['payment_id'], keep='first'
)
silver_payments = silver_payments[[
    'payment_id',
    'booking_id',
    'amount',
    'payment_method',
    '_ingested_at',
    '_source_file',
]]

In [26]:
# 7. PERSIST TO SILVER LAYER 
silver_flights.to_csv('storage/silver/flights/flights_silver.csv', index=False)
silver_passengers.to_csv(
    'storage/silver/passengers/passengers_silver.csv', index=False
)
silver_bookings.to_csv(
    'storage/silver/bookings/bookings_silver.csv', index=False
)
silver_payments.to_csv(
    'storage/silver/payments/payments_silver.csv', index=False
)

print('ALL 4 SILVER TABLES SUCCESSFULLY PROCESSED & SAVED ')

ALL 4 SILVER TABLES SUCCESSFULLY PROCESSED & SAVED 


## 2. Data Transformation and Cleaning

In [27]:
import hashlib
import os
import pandas as pd

In [28]:
# 1. SETUP SILVER STORAGE FOLDERS
for d in ['flights', 'passengers', 'bookings', 'payments']:
  os.makedirs(f'storage/silver/{d}', exist_ok=True)

In [29]:
# 2. PII MASKING FUNCTIONS
def hash_sha256(val):
  """Cryptographic SHA-256 hashing for unique sensitive identifiers."""
  if pd.isna(val) or str(val).strip() == '':
    return val
  return hashlib.sha256(str(val).strip().encode('utf-8')).hexdigest()

In [30]:
def mask_email(email):
  """Mask email username while maintaining domain format."""
  if pd.isna(email) or '@' not in str(email):
    return email
  parts = str(email).split('@')
  name = parts[0]
  masked_name = name[0] + '****' + name[-1] if len(name) > 2 else name[0] + '****'
  return f'{masked_name}@{parts[1]}'

In [31]:
def mask_phone(phone):
  """Mask phone number keeping only the last 4 visible digits."""
  p = str(phone).strip()
  if pd.isna(phone) or len(p) < 4:
    return phone
  return '****' + p[-4:]

In [32]:
# 3. LOAD BRONZE DATASETS
b_flights = pd.read_csv('storage/bronze/flights/flights_bronze.csv')
b_passengers = pd.read_csv('storage/bronze/passengers/passengers_bronze.csv')
b_bookings = pd.read_csv('storage/bronze/bookings/bookings_bronze.csv')
b_payments = pd.read_csv('storage/bronze/payments/payments_bronze.csv')

In [33]:
#  TRANSFORM 1: FLIGHTS 
b_flights['departure_time'] = pd.to_datetime(b_flights['departure_time'])
b_flights['arrival_time'] = pd.to_datetime(b_flights['arrival_time'])


def impute_airline(row):
  airline = str(row['airline']).strip()
  flight_id = str(row['flight_id']).strip()
  if pd.isna(row['airline']) or airline in ['UNKNOWN', 'nan', '']:
    if flight_id.startswith('AI'):
      return 'Air India'
    elif flight_id.startswith('6F') or flight_id.startswith('6E'):
      return 'IndiGo'
    elif flight_id.startswith('SJ'):
      return 'SpiceJet'
    elif flight_id.startswith('UK'):
      return 'Vistara'
    else:
      return 'Other'
  return airline


b_flights['airline_clean'] = b_flights.apply(impute_airline, axis=1)
overnight_mask = b_flights['arrival_time'] < b_flights['departure_time']
b_flights['arrival_time_clean'] = b_flights['arrival_time']
b_flights.loc[overnight_mask, 'arrival_time_clean'] += pd.Timedelta(days=1)
b_flights['duration_minutes'] = (
    b_flights['arrival_time_clean'] - b_flights['departure_time']
).dt.total_seconds() / 60.0

s_flights = b_flights.drop_duplicates(
    subset=['flight_id', 'departure_time'], keep='first'
).copy()
s_flights['airline'] = s_flights['airline_clean']
s_flights['arrival_time'] = s_flights['arrival_time_clean']
s_flights = s_flights[[
    'flight_id',
    'airline',
    'source',
    'destination',
    'departure_time',
    'arrival_time',
    'duration_minutes',
    '_ingested_at',
    '_source_file',
]]

In [34]:
#  TRANSFORM 2: PASSENGERS (WITH PII MASKING)
b_passengers['last_name'] = b_passengers['last_name'].fillna('Unknown')
b_passengers['date_of_birth'] = pd.to_datetime(b_passengers['date_of_birth'])
b_passengers['age'] = (
    pd.to_datetime('2026-09-05') - b_passengers['date_of_birth']
).dt.days // 365

b_passengers['aadhaar_id_hashed'] = b_passengers['aadhaar_id'].apply(
    hash_sha256
)
b_passengers['email_masked'] = b_passengers['email'].apply(mask_email)
b_passengers['phone_masked'] = b_passengers['phone'].apply(mask_phone)

s_passengers = b_passengers.drop_duplicates(
    subset=['passenger_id'], keep='first'
).copy()
s_passengers = s_passengers[[
    'passenger_id',
    'first_name',
    'last_name',
    'age',
    'gender',
    'email_masked',
    'phone_masked',
    'aadhaar_id_hashed',
    'date_of_birth',
    '_ingested_at',
    '_source_file',
]].rename(columns={
    'email_masked': 'email',
    'phone_masked': 'phone',
    'aadhaar_id_hashed': 'aadhaar_id',
})

In [35]:
#  TRANSFORM 3: BOOKINGS (WITH PII MASKING) 
s_bookings = b_bookings[b_bookings['status'] != 'INVALID'].copy()
s_bookings['status'] = s_bookings['status'].fillna('PENDING')
s_bookings['passport_number_hashed'] = s_bookings['passport_number'].apply(
    hash_sha256
)
s_bookings['emergency_contact_phone_masked'] = s_bookings[
    'emergency_contact_phone'
].apply(mask_phone)

s_bookings = s_bookings.drop_duplicates(
    subset=['booking_id'], keep='first'
).copy()
s_bookings = s_bookings[[
    'booking_id',
    'passenger_id',
    'flight_id',
    'booking_date',
    'status',
    'passport_number_hashed',
    'seat_number',
    'emergency_contact_name',
    'emergency_contact_phone_masked',
    '_ingested_at',
    '_source_file',
]].rename(columns={
    'passport_number_hashed': 'passport_number',
    'emergency_contact_phone_masked': 'emergency_contact_phone',
})

In [36]:
#  TRANSFORM 4: PAYMENTS
b_payments['amount_numeric'] = pd.to_numeric(
    b_payments['amount'], errors='coerce'
)
s_payments = b_payments[~b_payments['amount_numeric'].isnull()].copy()
s_payments['amount'] = s_payments['amount_numeric']
s_payments = s_payments.drop_duplicates(subset=['payment_id'], keep='first')
s_payments = s_payments[[
    'payment_id',
    'booking_id',
    'amount',
    'payment_method',
    '_ingested_at',
    '_source_file',
]]

In [38]:
#  PERSIST ALL SILVER CLEAN & MASKED TABLES 
s_flights.to_csv('storage/silver/flights/flights_silver.csv', index=False)
s_passengers.to_csv(
    'storage/silver/passengers/passengers_silver.csv', index=False
)
s_bookings.to_csv('storage/silver/bookings/bookings_silver.csv', index=False)
s_payments.to_csv('storage/silver/payments/payments_silver.csv', index=False)

print(' DATA TRANSFORMATION & PII MASKING COMPLETED SUCCESSFULLY')

 DATA TRANSFORMATION & PII MASKING COMPLETED SUCCESSFULLY


### 3. Data Modelling, Storage and BusineKPIs 

In [39]:
import os
import pandas as pd

In [40]:
# 1. SETUP GOLD STORAGE DIRECTORY
os.makedirs('storage/gold', exist_ok=True)

In [41]:
# 2. LOAD CLEANED SILVER DATASETS
s_flights = pd.read_csv('storage/silver/flights/flights_silver.csv')
s_bookings = pd.read_csv('storage/silver/bookings/bookings_silver.csv')
s_passengers = pd.read_csv('storage/silver/passengers/passengers_silver.csv')
s_payments = pd.read_csv('storage/silver/payments/payments_silver.csv')

# Pre-calculate Route
s_flights['route'] = s_flights['source'] + ' -> ' + s_flights['destination']

In [47]:
# 3. PREPARE CLEAN DIMENSION TABLES (Remove Metadata Attributes)
dim_flights = s_flights.drop(
    columns=['_ingested_at', '_source_file'], errors='ignore'
)
dim_passengers = s_passengers.drop(
    columns=['_ingested_at', '_source_file'], errors='ignore'
)
dim_payments = s_payments.drop(
    columns=['_ingested_at', '_source_file'], errors='ignore'
)

In [48]:
# 4. PERSIST ALL DIMENSION TABLES TO GOLD STORAGE
dim_flights.to_csv('storage/gold/dim_flights.csv', index=False)
dim_passengers.to_csv('storage/gold/dim_passengers.csv', index=False)
dim_payments.to_csv('storage/gold/dim_payments.csv', index=False)

In [49]:
# 5. BUILD CONSOLIDATED FACT TABLE (fact_bookings)
fact_bookings = (
    s_bookings.drop(columns=['_ingested_at', '_source_file'], errors='ignore')
    .merge(dim_payments, on='booking_id', how='left')
    .merge(dim_flights, on='flight_id', how='left')
    .merge(dim_passengers, on='passenger_id', how='left')
)

fact_bookings.to_csv('storage/gold/fact_bookings.csv', index=False)

In [50]:
# 6. COMPUTE BUSINESS KPIS & ANALYTICS
avg_duration_overall = dim_flights['duration_minutes'].mean()
avg_duration_by_airline = (
    dim_flights.groupby('airline')['duration_minutes']
    .mean()
    .reset_index()
    .rename(columns={'duration_minutes': 'avg_duration_mins'})
)

route_flights = (
    dim_flights.groupby('route')
    .size()
    .reset_index(name='total_scheduled_flights')
)
route_passengers = (
    fact_bookings.groupby('route')
    .size()
    .reset_index(name='total_passenger_bookings')
)
route_traffic = route_flights.merge(
    route_passengers, on='route', how='left'
).sort_values(by='total_passenger_bookings', ascending=False)

airline_dist = dim_flights['airline'].value_counts().reset_index()
airline_dist.columns = ['airline', 'flight_count']
airline_dist['market_share_pct'] = (
    airline_dist['flight_count'] / len(dim_flights)
) * 100

total_revenue = dim_payments['amount'].sum()
avg_ticket_price = dim_payments['amount'].mean()

print('GOLD LAYER STAR SCHEMA & KPIS GENERATED SUCCESSFULLY')
print(
    f'Overall Avg Duration: {avg_duration_overall:.2f} mins'
    f' ({avg_duration_overall/60:.2f} hrs)'
)
print(f'Total Gross Revenue: ₹{total_revenue:,.2f}')
print(f'Average Ticket Price: ₹{avg_ticket_price:,.2f}')

GOLD LAYER STAR SCHEMA & KPIS GENERATED SUCCESSFULLY
Overall Avg Duration: 164.62 mins (2.74 hrs)
Total Gross Revenue: ₹7,385,142.98
Average Ticket Price: ₹8,009.92
